## Imports

In [ ]:
import simpy
import random
import numpy as np
import pandas as pd
import copy
from scipy import stats
from scipy.integrate import solve_ivp
from collections import defaultdict
import statsmodels.api as sm
import statsmodels.formula.api as smf

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec
plt.switch_backend('TkAgg')
import matplotlib.animation as animation
from matplotlib.widgets import Slider, Button, RadioButtons

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# from warehouse_pinn_simulation import DegradationPINN, generate_synthetic_degradation_data, train_degradation_model #(if standalone script for a headless run)
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RANSACRegressor

## Simulation Demo

In [ ]:

# ==============================
# Global Simulation Parameters
# ==============================
SIMULATION_TIME = 28800  # 8 hours in seconds
WINDOW_SIZE = 1800  # 30-minute sliding window for throughput
BATCH_WINDOW = 120  # 2-minute window (for wave picking)

# Equipment speeds and times
CRANE_VERTICAL_SPEED = 1.0
CRANE_HORIZONTAL_SPEED = 1.0
CRANE_ACCEL_DECEL_TIME = 1.2
SHUTTLE_SPEED = 3.0
SHUTTLE_ACCEL_DECEL_TIME = 0.8

# Cost rates
AUTOMATED_COST_RATE = 1.0
CROSS_DOCK_COST_RATE = 0.8

# Other parameters
REPLENISH_INTERVAL = 900  # every 15 minutes
REORGANIZE_INTERVAL = 1800  # every 30 minutes
UNCERTAINTY_MEAN = 0.0
UNCERTAINTY_SIGMA = 0.2
FAILURE_RATE = 0.01

# ------------------
# Batch Picking Real-World Data
# ------------------
def generate_batch_size():
    if random.random() < 0.38:
        return 10
    else:
        return 3 if random.random() < 0.45 else 4

MAX_ITEMS_CAPACITY = 35

# ------------------
# ABC Curve Storage Parameters
# ------------------
# 4 different storage policy scenarios
STORAGE_POLICIES = {
    "20_70": {"name": "20%/70% curve", "description": "20% SKUs account for 70% of activity"},
    "20_90": {"name": "20%/90% curve", "description": "20% SKUs account for 90% of activity"},
    "20_50": {"name": "20%/50% curve", "description": "20% SKUs account for 50% of activity"},
    "20_20": {"name": "20%/20% curve", "description": "20% SKUs account for 20% of activity"}
}

current_storage_policy = "20_70"  # Default policy

# ------------------
# Product Types Definition (Expanded to 11 SKUs)
# ------------------
# Function to generate the product types dictionary based on storage policy
def generate_product_types(policy=None):
    if policy is None:
        policy = current_storage_policy
    
    # Create basic structure for 11 distinct SKUs
    product_types = {}
    
    # Determine weights based on the selected policy
    a_skus = 2     # ~20% of the 11 SKUs are 'A' class
    b_skus = 4     # ~40% of SKUs are 'B' class
    c_skus = 5     # ~40% of SKUs are 'C' class
    
    # Generate weights for each ABC class based on the storage policy
    if policy == "20_70":
        # 20/70: A items (20%) account for 70% of activity
        a_weight = 0.70 / a_skus
        b_weight = 0.20 / b_skus
        c_weight = 0.10 / c_skus
    elif policy == "20_90":
        # 20/90: A items (20%) account for 90% of activity
        a_weight = 0.90 / a_skus
        b_weight = 0.08 / b_skus
        c_weight = 0.02 / c_skus
    elif policy == "20_50":
        # 20/50: A items (20%) account for 50% of activity
        a_weight = 0.50 / a_skus
        b_weight = 0.30 / b_skus
        c_weight = 0.20 / c_skus
    else:  # "20_20"
        # 20/20: A items (20%) account for 20% of activity (equal distribution)
        a_weight = 0.20 / a_skus
        b_weight = 0.35 / b_skus
        c_weight = 0.45 / c_skus
    
    # Create product definitions
    # A-class products (highest activity)
    for i in range(a_skus):
        product_types[f"A{i+1}"] = {
            "class": "A",
            "priority": 0,
            "processing_multiplier": 1.0,
            "travel_factor": 1.0,
            "activity_weight": a_weight
        }
    
    # B-class products (medium activity)
    for i in range(b_skus):
        product_types[f"B{i+1}"] = {
            "class": "B",
            "priority": 1,
            "processing_multiplier": 1.2,
            "travel_factor": 1.1,
            "activity_weight": b_weight
        }
    
    # C-class products (lowest activity)
    for i in range(c_skus):
        product_types[f"C{i+1}"] = {
            "class": "C",
            "priority": 2,
            "processing_multiplier": 1.5,
            "travel_factor": 1.3,
            "activity_weight": c_weight
        }
    
    return product_types

# Initialize the product types
PRODUCT_TYPES = generate_product_types()

# Function to randomly select a product based on activity weights
def select_weighted_product():
    products = list(PRODUCT_TYPES.keys())
    weights = [PRODUCT_TYPES[p]["activity_weight"] for p in products]
    return random.choices(products, weights=weights, k=1)[0]

# ------------------
# n-class continuous-space storage implementation
# ------------------
def calculate_product_storage_location(product_code, num_aisles):
    """
    Determines optimal storage location based on product class and ABC curve
    For n-class continuous-space storage, we allocate space proportionally based on activity
    """
    product_class = PRODUCT_TYPES[product_code]["class"]
    
    if product_class == "A":
        # A-class items stored closest to I/O point (first 20% of aisles)
        max_aisle = max(1, int(0.2 * num_aisles))
        return list(range(max_aisle))
    elif product_class == "B":
        # B-class items stored in middle (next 30% of aisles)
        start_aisle = max(1, int(0.2 * num_aisles))
        end_aisle = max(start_aisle + 1, int(0.5 * num_aisles))
        return list(range(start_aisle, end_aisle))
    else:  # C class
        # C-class items stored furthest (remaining 50% of aisles)
        start_aisle = max(1, int(0.5 * num_aisles))
        return list(range(start_aisle, num_aisles))

# ------------------
# Puzzle-based Retrieval Functions (for cube-based)
# ------------------
def puzzle_moves_single_escort(i, j):
    if i == 1 and j == 1:
        return 0
    if i > j:
        return 6 * i + 2 * j - 13
    elif j > i:
        return 6 * j + 2 * i - 13
    else:
        return 8 * i - 11

def puzzle_moves_multi_escort(i, j, num_escorts):
    base = puzzle_moves_single_escort(i, j)
    improvement_factor = 1.0 - min(0.6, 0.1 * (num_escorts - 1))
    return max(0, int(round(base * improvement_factor)))

# ------------------
# Order Arrival Rate Function
# ------------------
def get_poisson_arrival_rate(current_time):
    # Define lambda in orders per second
    if current_time < 14400:  # first 4 hours
        lambda_per_sec = 200.0 / 3600.0
    else:
        lambda_per_sec = 150.0 / 3600.0
    return lambda_per_sec

# ==============================
# Warehouse Simulation Class
# ==============================
class Warehouse:
    def __init__(self, env, storage_system_type, num_cranes, num_shuttles, num_robots,
                 cross_docking_rate, peak_multiplier, deep_config, command_cycle,
                 num_aisles, dynamic_storage, batch_mode, wave_mode, storage_policy=None):
        self.env = env
        self.storage_system_type = storage_system_type
        self.cross_docking_rate = cross_docking_rate
        self.peak_multiplier = peak_multiplier
        self.deep_config = deep_config
        self.command_cycle = command_cycle
        self.num_aisles = num_aisles
        self.dynamic_storage = dynamic_storage
        self.batch_mode = batch_mode
        self.wave_mode = wave_mode
        self.storage_policy = storage_policy if storage_policy else current_storage_policy
        
        # Update product types based on storage policy
        global PRODUCT_TYPES
        PRODUCT_TYPES = generate_product_types(self.storage_policy)
        
        if self.storage_system_type == "cube_based":
            self.puzzle_width = self.num_aisles
            self.puzzle_height = self.num_aisles
            self.num_escorts = num_robots
        
        if self.storage_system_type in ["unit_load", "miniload"]:
            self.cranes = [simpy.PriorityResource(env, capacity=1) for _ in range(num_aisles)]
            self.elevators = self.shuttles = self.robots = None
        elif self.storage_system_type == "shuttle_based":
            self.elevators = [simpy.PriorityResource(env, capacity=1) for _ in range(num_aisles)]
            self.shuttles = [simpy.PriorityResource(env, capacity=1) for _ in range(num_aisles)]
            self.cranes = self.robots = None
        elif self.storage_system_type == "cube_based":
            per_aisle = max(1, num_robots // num_aisles)
            self.robots = [simpy.PriorityResource(env, capacity=per_aisle) for _ in range(num_aisles)]
            self.cranes = self.elevators = self.shuttles = None
        
        # For dynamic storage, we'll track products by class (A, B, C) instead of individual products
        if self.dynamic_storage:
            self.storage_map = {
                "A": list(range(int(0.2 * num_aisles))),
                "B": list(range(int(0.2 * num_aisles), int(0.5 * num_aisles))),
                "C": list(range(int(0.5 * num_aisles), num_aisles))
            }
        
        # Statistics tracking
        self.orders_processed = 0
        self.total_cost = 0
        self.order_completion_times = []  # Total time from order arrival to completion
        self.order_wait_times = []        # Total waiting time (time spent waiting for resources)
        self.order_fulfillment_times = [] # This is the same as completion_time - arrival; recorded separately
        self.queue_delays = []
        self.batch_sizes = []
        self.batch_pool = []
        self.sliding_win_throughput = []
        self.resource_usage = {"cranes": [], "elevators": [], "shuttles": [], "robots": []}
        
        # Additional tracking for SKU distribution
        self.sku_counts = {sku: 0 for sku in PRODUCT_TYPES.keys()}
        
        # Start generator processes
        self.env.process(self.order_generator())
        if self.batch_mode and self.wave_mode:
            self.env.process(self.wave_picker())
        self.env.process(self.calculate_sliding_window_throughput())
        self.env.process(self.replenishment_process())
        if self.dynamic_storage:
            self.env.process(self.reorganization_process())
        self.env.process(self.track_utilization())
    
    def get_product_storage(self, product_type):
        if self.dynamic_storage:
            # Get product class (A, B, or C)
            product_class = PRODUCT_TYPES[product_type]["class"]
            return self.storage_map.get(product_class, list(range(self.num_aisles)))
        else:
            # Use the continuous space allocation based on product class
            return calculate_product_storage_location(product_type, self.num_aisles)
    
    def get_adjusted_priority(self, product):
        base_priority = PRODUCT_TYPES[product]["priority"]
        return max(0, base_priority - 1) if self.env.now > SIMULATION_TIME * 0.75 else base_priority
    
    def order_generator(self):
        while True:
            # Generate order with 11 distinct SKUs
            num_items = random.randint(2, 4)
            products = [select_weighted_product() for _ in range(num_items)]
            
            # Track SKU distribution
            for product in products:
                self.sku_counts[product] += 1
                
            order = {"arrival_time": self.env.now, "products": products, "num_items": num_items}
            
            if self.batch_mode:
                self.batch_pool.append(order)
                if not self.wave_mode:
                    target_batch_size = generate_batch_size()
                    total_items = sum(o["num_items"] for o in self.batch_pool)
                    if len(self.batch_pool) >= target_batch_size or total_items >= MAX_ITEMS_CAPACITY:
                        current_batch = list(self.batch_pool)
                        self.batch_pool.clear()
                        self.batch_sizes.append(len(current_batch))
                        for ord in current_batch:
                            self.env.process(self.process_order(ord))
            else:
                self.env.process(self.process_order(order))
                
            lambda_rate = get_poisson_arrival_rate(self.env.now) * self.peak_multiplier
            yield self.env.timeout(np.random.exponential(1.0 / lambda_rate))
    
    def wave_picker(self):
        while True:
            yield self.env.timeout(BATCH_WINDOW)
            if self.batch_pool:
                current_batch = list(self.batch_pool)
                self.batch_pool.clear()
                self.batch_sizes.append(len(current_batch))
                for ord in current_batch:
                    self.env.process(self.process_order(ord))
    
    def process_order(self, order):
        arrival = order["arrival_time"]
        order_wait = 0  # Accumulate wait time across all products in this order
        for product in order["products"]:
            if random.random() < 0.1:
                wait = yield from self.process_cross_dock(product)
            else:
                if self.storage_system_type == "unit_load":
                    wait = yield from self.process_unit_load(product)
                elif self.storage_system_type == "miniload":
                    wait = yield from self.process_miniload(product)
                elif self.storage_system_type == "shuttle_based":
                    wait = yield from self.process_shuttle_based(product)
                elif self.storage_system_type == "cube_based":
                    wait = yield from self.process_cube_based(product)
            order_wait += wait
        self.orders_processed += 1
        fulfillment_time = self.env.now - arrival
        self.order_fulfillment_times.append(fulfillment_time)
        self.order_completion_times.append(self.env.now)
        self.order_wait_times.append(order_wait)
    
    def process_cross_dock(self, product):
        base_time = random.uniform(2, 4) * PRODUCT_TYPES[product]["processing_multiplier"]
        yield self.env.timeout(base_time)
        self.total_cost += base_time * CROSS_DOCK_COST_RATE
        return 0
    
    def process_unit_load(self, product):
        total_wait = 0
        aisles = self.get_product_storage(product)
        best = min(aisles, key=lambda i: len(self.cranes[i].users) + len(self.cranes[i].queue))
        request_time = self.env.now
        with self.cranes[best].request(priority=self.get_adjusted_priority(product)) as req:
            yield req
            service_start_time = self.env.now
            wait_time = service_start_time - request_time
            total_wait += wait_time
            self.queue_delays.append(wait_time)
        
            tf = PRODUCT_TYPES[product]["travel_factor"]
            # Unit‑load is heavier/slower, with longer travel distances
            v_dist = np.random.uniform(2, 4) * tf
            h_dist = np.random.uniform(4, 7) * tf
            crane_speed_multiplier = 0.8  # slower movement
            failure_chance = FAILURE_RATE * 1.5
            downtime_min, downtime_max = 20, 40
            
            # compute base_time using the adjusted speed
            base_time = (
                v_dist / (CRANE_HORIZONTAL_SPEED * crane_speed_multiplier)
                + h_dist / (CRANE_VERTICAL_SPEED * crane_speed_multiplier)
                + CRANE_ACCEL_DECEL_TIME
            )
            base_time *= PRODUCT_TYPES[product]["processing_multiplier"]
            base_time *= np.random.lognormal(UNCERTAINTY_MEAN, UNCERTAINTY_SIGMA)
            
            yield self.env.timeout(base_time)
            self.total_cost += base_time * AUTOMATED_COST_RATE
            
            # now simulate type‑specific failures
            if random.random() < failure_chance:
                downtime = random.uniform(downtime_min, downtime_max)
                yield self.env.timeout(downtime)
                self.total_cost += downtime * AUTOMATED_COST_RATE
        return total_wait
    
    def process_miniload(self, product):
        total_wait = 0
        aisles = self.get_product_storage(product)
        best = min(aisles, key=lambda i: len(self.cranes[i].users) + len(self.cranes[i].queue))
        request_time = self.env.now
        with self.cranes[best].request(priority=self.get_adjusted_priority(product)) as req:
            yield req
            service_start_time = self.env.now
            wait_time = service_start_time - request_time
            total_wait += wait_time
            self.queue_delays.append(wait_time)
        
            tf = PRODUCT_TYPES[product]["travel_factor"]
            # Miniload is lighter/faster with shorter travel distances
            v_dist = np.random.uniform(1, 2) * tf
            h_dist = np.random.uniform(2, 4) * tf
            crane_speed_multiplier = 1.2  # faster movement
            failure_chance = FAILURE_RATE * 0.8
            downtime_min, downtime_max = 5, 15
            
            base_time = (
                v_dist / (CRANE_VERTICAL_SPEED * crane_speed_multiplier)
                + h_dist / (CRANE_HORIZONTAL_SPEED * crane_speed_multiplier)
                + CRANE_ACCEL_DECEL_TIME
            )
            base_time *= PRODUCT_TYPES[product]["processing_multiplier"]
            base_time *= np.random.lognormal(UNCERTAINTY_MEAN, UNCERTAINTY_SIGMA)
            
            yield self.env.timeout(base_time)
            self.total_cost += base_time * AUTOMATED_COST_RATE
            
            if random.random() < failure_chance:
                downtime = random.uniform(downtime_min, downtime_max)
                yield self.env.timeout(downtime)
                self.total_cost += downtime * AUTOMATED_COST_RATE
        return total_wait
    
    def process_shuttle_based(self, product):
        total_wait = 0
        aisles = self.get_product_storage(product)
        # Elevator phase
        best_elev = min(aisles, key=lambda i: len(self.elevators[i].users) + len(self.elevators[i].queue))
        request_time = self.env.now
        with self.elevators[best_elev].request(priority=self.get_adjusted_priority(product)) as req:
            yield req
            service_start_time = self.env.now
            wait_elev = service_start_time - request_time
            total_wait += wait_elev
            self.queue_delays.append(wait_elev)
            tf = PRODUCT_TYPES[product]["travel_factor"]
            v_dist = np.random.uniform(1, 3) * tf
            elev_time = (v_dist / CRANE_VERTICAL_SPEED + CRANE_ACCEL_DECEL_TIME)
            if self.deep_config == "double":
                elev_time *= 1.1
            if self.command_cycle == "dual":
                elev_time *= 1.05
            elev_time *= PRODUCT_TYPES[product]["processing_multiplier"]
            elev_time *= np.random.lognormal(UNCERTAINTY_MEAN, UNCERTAINTY_SIGMA)
            yield from self.simulate_process(elev_time)
        # Shuttle phase
        best_shut = min(aisles, key=lambda i: len(self.shuttles[i].users) + len(self.shuttles[i].queue))
        request_time = self.env.now
        with self.shuttles[best_shut].request(priority=self.get_adjusted_priority(product)) as req:
            yield req
            service_start_time = self.env.now
            wait_shut = service_start_time - request_time
            total_wait += wait_shut
            self.queue_delays.append(wait_shut)
            tf = PRODUCT_TYPES[product]["travel_factor"]
            h_dist = np.random.uniform(2, 4) * tf
            shut_time = (h_dist / SHUTTLE_SPEED + SHUTTLE_ACCEL_DECEL_TIME)
            if self.deep_config == "double":
                shut_time *= 1.05
            if self.command_cycle == "dual":
                shut_time *= 1.05
            shut_time *= PRODUCT_TYPES[product]["processing_multiplier"]
            shut_time *= np.random.lognormal(UNCERTAINTY_MEAN, UNCERTAINTY_SIGMA)
            yield from self.simulate_process(shut_time)
        return total_wait
    
    def process_cube_based(self, product):
        total_wait = 0
        aisles = self.get_product_storage(product)
        best_aisle = min(aisles, key=lambda i: len(self.robots[i].users) + len(self.robots[i].queue))
        i = random.randint(1, self.puzzle_height)
        j = random.randint(1, self.puzzle_width)
        request_time = self.env.now
        with self.robots[best_aisle].request(priority=self.get_adjusted_priority(product)) as req:
            yield req
            service_start_time = self.env.now
            wait_time = service_start_time - request_time
            total_wait += wait_time
            self.queue_delays.append(wait_time)
            e = self.num_escorts if hasattr(self, 'num_escorts') else 1
            if e <= 1:
                puzzle_moves = puzzle_moves_single_escort(i, j)
            else:
                puzzle_moves = puzzle_moves_multi_escort(i, j, e)
            base_time = float(puzzle_moves)
            base_time *= PRODUCT_TYPES[product]["processing_multiplier"]
            base_time *= np.random.lognormal(UNCERTAINTY_MEAN, UNCERTAINTY_SIGMA)
            yield self.env.timeout(base_time)
            self.total_cost += base_time * AUTOMATED_COST_RATE
        return total_wait
    
    def simulate_process(self, duration):
        yield self.env.timeout(duration)
        self.total_cost += duration * AUTOMATED_COST_RATE
        if random.random() < FAILURE_RATE:
            downtime = random.uniform(10, 30)
            yield self.env.timeout(downtime)
            self.total_cost += downtime * AUTOMATED_COST_RATE
    
    def replenishment_process(self):
        while True:
            yield self.env.timeout(REPLENISH_INTERVAL)
            if self.storage_system_type in ["unit_load", "miniload"]:
                for r in self.cranes:
                    self.env.process(self.fake_replenish(r))
            elif self.storage_system_type == "shuttle_based":
                for elev, shut in zip(self.elevators, self.shuttles):
                    self.env.process(self.fake_replenish(elev))
                    self.env.process(self.fake_replenish(shut))
            elif self.storage_system_type == "cube_based":
                for r in self.robots:
                    self.env.process(self.fake_replenish(r))
    
    def fake_replenish(self, resource):
        with resource.request() as req:
            yield req
            yield self.env.timeout(random.uniform(5, 10))
    
    def reorganization_process(self):
        while True:
            yield self.env.timeout(REORGANIZE_INTERVAL)
            # Reorganize storage by class rather than by individual product
            for class_key in self.storage_map:
                random.shuffle(self.storage_map[class_key])
    
    def calculate_sliding_window_throughput(self):
        while True:
            now = self.env.now
            count = sum(1 for t in self.order_completion_times if now - WINDOW_SIZE <= t <= now)
            hourly_rate = count * (3600.0 / WINDOW_SIZE)
            self.sliding_win_throughput.append((now, hourly_rate))
            yield self.env.timeout(WINDOW_SIZE / 4)
    
    def track_utilization(self):
        while True:
            if self.storage_system_type in ["unit_load", "miniload"]:
                usage = sum(len(c.users) for c in self.cranes)
                queue = sum(len(c.queue) for c in self.cranes)
                self.resource_usage["cranes"].append((usage, queue))
            elif self.storage_system_type == "shuttle_based":
                usage = sum(len(e.users) for e in self.elevators)
                queue = sum(len(e.queue) for e in self.elevators)
                self.resource_usage["elevators"].append((usage, queue))
            elif self.storage_system_type == "cube_based":
                usage = sum(len(r.users) for r in self.robots)
                queue = sum(len(r.queue) for r in self.robots)
                self.resource_usage["robots"].append((usage, queue))
            yield self.env.timeout(10)

# ==============================
# Simulation Function
# ==============================
def run_simulation(storage_system_type, num_cranes, num_shuttles, num_robots, num_aisles,
                  cross_docking_rate, peak_multiplier, deep_config, command_cycle,
                  dynamic_storage, batch_mode, wave_mode=False, storage_policy="20_70", seed=12345):
    random.seed(seed)
    np.random.seed(seed)
    env = simpy.Environment()
    warehouse = Warehouse(env, storage_system_type, num_cranes, num_shuttles, num_robots,
                        cross_docking_rate, peak_multiplier, deep_config, command_cycle,
                        num_aisles, dynamic_storage, batch_mode, wave_mode, storage_policy)
    env.run(until=SIMULATION_TIME)
    
    throughput = warehouse.orders_processed / (SIMULATION_TIME / 3600.0)
    avg_fulfillment_time = np.mean(warehouse.order_fulfillment_times) if warehouse.order_fulfillment_times else 0.0
    avg_wait_time = np.mean(warehouse.order_wait_times) if warehouse.order_wait_times else 0.0
    
    return (throughput, warehouse.resource_usage, warehouse.total_cost,
            warehouse.sliding_win_throughput, avg_fulfillment_time, avg_wait_time,
            warehouse.order_completion_times, warehouse.order_wait_times,
            warehouse.batch_sizes if batch_mode else None, warehouse)

# ==============================
# Interactive Visualization Dashboard
# ==============================
def create_interactive_dashboard():
    fig = plt.figure(figsize=(14, 10))
    gs_left = gridspec.GridSpec(4, 1, left=0.05, right=0.60, top=0.95, bottom=0.05, hspace=0.35)
    ax_bar = fig.add_subplot(gs_left[0, 0])
    ax_slide = fig.add_subplot(gs_left[1, 0])
    ax_resource = fig.add_subplot(gs_left[2, 0])
    ax_text = fig.add_subplot(gs_left[3, 0])
    
    slider_height = 0.04
    slider_gap = 0.01
    right_left = 0.72
    right_width = 0.22
    start_y = 0.95 - slider_height
    
    slider_cranes_ax = fig.add_axes([right_left, start_y, right_width, slider_height])
    slider_shuttles_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*1, right_width, slider_height])
    slider_robots_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*2, right_width, slider_height])
    slider_aisles_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*3, right_width, slider_height])
    slider_deep_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*4, right_width, slider_height])
    slider_command_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*5, right_width, slider_height])
    slider_peak_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*6, right_width, slider_height])
    slider_storage_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*7, right_width, slider_height])
    slider_batch_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*8, right_width, slider_height])
    slider_display_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*9, right_width, slider_height])
    
    # Radio button for ABC policy selection in a 2x2 grid
    policy_radio_ax = fig.add_axes([right_left, start_y - (slider_height+slider_gap)*12.75, right_width, slider_height*4])
    policy_labels = ['20%/70%', '20%/90%', '20%/50%', '20%/20%']
    
    # Create radio buttons
    policy_radio = RadioButtons(policy_radio_ax, policy_labels, activecolor='green')
    policy_radio_ax.set_title('ABC Policy', fontsize=10)
    
    slider_cranes = Slider(slider_cranes_ax, 'Cranes', 1, 20, valinit=10, valstep=1)
    slider_shuttles = Slider(slider_shuttles_ax, 'Shuttles', 1, 20, valinit=0, valstep=1)
    slider_robots = Slider(slider_robots_ax, 'Robots', 1, 20, valinit=0, valstep=1)
    slider_aisles = Slider(slider_aisles_ax, 'Aisles', 3, 20, valinit=10, valstep=1)
    slider_deep = Slider(slider_deep_ax, 'Depth (1=single,2=double)', 1, 2, valinit=1, valstep=1)
    slider_command = Slider(slider_command_ax, 'Command (1=single,2=dual)', 1, 2, valinit=1, valstep=1)
    slider_peak = Slider(slider_peak_ax, 'Peak Mult.', 1.0, 3.0, valinit=1.0, valstep=0.1)
    slider_storage = Slider(slider_storage_ax, 'Storage Mode (0=Static,1=Dynamic)', 0, 1, valinit=0, valstep=1)
    slider_batch = Slider(slider_batch_ax, 'Batch Mode (0=No,1=Yes)', 0, 1, valinit=0, valstep=1)
    slider_display = Slider(slider_display_ax, 'Display (0=Both,1=Usage,2=Queue)', 0, 2, valinit=0, valstep=1)
    
    btn_ul_ax = fig.add_axes([right_left, 0.20, 0.08, 0.05])
    btn_ml_ax = fig.add_axes([right_left+0.09, 0.20, 0.08, 0.05])
    btn_sb_ax = fig.add_axes([right_left, 0.10, 0.08, 0.05])
    btn_cb_ax = fig.add_axes([right_left+0.09, 0.10, 0.08, 0.05])
    
    btn_ul = Button(btn_ul_ax, 'Unit-load')
    btn_ml = Button(btn_ml_ax, 'Miniload')
    btn_sb = Button(btn_sb_ax, 'Shuttle-based')
    btn_cb = Button(btn_cb_ax, 'Cube-based')
    
    storage_system_type = "unit_load"
    sim_results = {}
    
    def update(_=None):
        nonlocal storage_system_type, sim_results
        ax_bar.cla()
        ax_slide.cla()
        ax_resource.cla()
        ax_text.cla()
        
        current_cranes = int(slider_cranes.val)
        current_shuttles = int(slider_shuttles.val)
        current_robots = int(slider_robots.val)
        current_aisles = int(slider_aisles.val)
        deep_val = int(slider_deep.val)
        command_val = int(slider_command.val)
        current_peak = float(slider_peak.val)
        storage_mode = bool(int(slider_storage.val))
        batch_mode = bool(int(slider_batch.val))
        display_toggle = int(slider_display.val)
        # Get selected policy from radio buttons
        selected_policy = policy_radio.value_selected
        policy_map = {
            '20%/70%': "20_70",
            '20%/90%': "20_90", 
            '20%/50%': "20_50", 
            '20%/20%': "20_20"
        }
        current_policy = policy_map[selected_policy]
        
        deep_config = "double" if deep_val == 2 else "single"
        command_cycle = "dual" if command_val == 2 else "single"
        if deep_config == "double" and command_cycle == "single":
            command_cycle = "dual"
            slider_command.set_val(2)
        
        (throughput, resource_usage, total_cost, sliding_win_throughput,
         avg_fulfillment_time, avg_wait_time, completion_times, wait_times, batch_sizes,
         warehouse) = run_simulation(
            storage_system_type=storage_system_type,
            num_cranes=current_cranes,
            num_shuttles=current_shuttles,
            num_robots=current_robots,
            num_aisles=current_aisles,
            cross_docking_rate=CROSS_DOCK_COST_RATE,
            deep_config=deep_config,
            command_cycle=command_cycle,
            peak_multiplier=current_peak,
            dynamic_storage=storage_mode,
            batch_mode=batch_mode,
            storage_policy=current_policy
        )
        
        sim_results = {
            "throughput": throughput,
            "total_cost": total_cost,
            "avg_fulfillment_time": avg_fulfillment_time,
            "avg_wait": avg_wait_time,
            "avg_batch": np.mean(batch_sizes) if (batch_mode and batch_sizes) else None,
            "avg_queue_delay": np.mean(warehouse.queue_delays) if warehouse.queue_delays else 0.0
        }
        
        # Calculate SKU distribution metrics for display
        sku_counts = warehouse.sku_counts
        total_picks = sum(sku_counts.values())
        
        # Group products by class
        class_picks = {"A": 0, "B": 0, "C": 0}
        for sku, count in sku_counts.items():
            product_class = PRODUCT_TYPES[sku]["class"]
            class_picks[product_class] += count
            
        class_percentages = {
            cls: (count / total_picks * 100 if total_picks > 0 else 0) 
            for cls, count in class_picks.items()
        }
        
        num_windows = int(SIMULATION_TIME / WINDOW_SIZE)
        binned_throughput = []
        x_tick_positions = [0]
        x_tick_labels = ["0.0"]
        for i in range(num_windows):
            start = i * WINDOW_SIZE
            end = (i + 1) * WINDOW_SIZE
            count = sum(1 for t in completion_times if start <= t < end)
            hourly_rate = count * (3600 // WINDOW_SIZE)
            binned_throughput.append(hourly_rate)
            end_hour = end / 3600
            x_tick_positions.append(i + 1)
            x_tick_labels.append(f"{end_hour:.1f}")
        
        ax_bar.bar(range(num_windows), binned_throughput, width=1.0, align='edge', alpha=0.7)
        ax_bar.set_title(f"Throughput ({storage_system_type}, {deep_config}, {command_cycle}, {STORAGE_POLICIES[current_policy]['name']})")
        ax_bar.set_xlabel("Time (hours)")
        ax_bar.set_ylabel("Orders/hr")
        ax_bar.set_xticks(x_tick_positions)
        ax_bar.set_xticklabels(x_tick_labels)
        ax_bar.text(0.95, 0.95, f"Total Cost: {total_cost:.1f}", transform=ax_bar.transAxes,
                  ha="right", va="top", bbox=dict(facecolor='white', alpha=0.0))
        
        times = [pt[0] for pt in sliding_win_throughput]
        rates = [pt[1] for pt in sliding_win_throughput]
        ax_slide.plot(times, rates, '-o', color='green', alpha=0.7)
        ax_slide.set_title("Sliding-Window Throughput")
        ax_slide.set_xlabel("Simulation Time (s)")
        ax_slide.set_ylabel("Orders/hr")
        
        if storage_system_type in ["unit_load", "miniload"]:
            usage_vals = [u for (u, q) in resource_usage["cranes"]]
            queue_vals = [q for (u, q) in resource_usage["cranes"]]
        elif storage_system_type == "shuttle_based":
            usage_vals = [u for (u, q) in resource_usage["elevators"]]
            queue_vals = [q for (u, q) in resource_usage["elevators"]]
        elif storage_system_type == "cube_based":
            usage_vals = [u for (u, q) in resource_usage["robots"]]
            queue_vals = [q for (u, q) in resource_usage["robots"]]
        else:
            usage_vals, queue_vals = [], []
        
        if display_toggle == 0:
            ax_resource.plot(usage_vals, label="Resource Usage")
            ax_resource.plot(queue_vals, label="Queue Length", linestyle="--")
            ax_resource.set_title("Resource Usage & Queue")
        elif display_toggle == 1:
            ax_resource.plot(usage_vals, label="Resource Usage")
            ax_resource.set_title("Resource Usage Over Time")
        else:
            ax_resource.plot(queue_vals, label="Queue Length", linestyle="--")
            ax_resource.set_title("Resource Queue Over Time")
        
        ax_resource.set_xlabel("Time (steps)")
        ax_resource.set_ylabel("Count")
        ax_resource.legend()
        
        text_lines = [
            f"Throughput = {sim_results['throughput']:.1f} orders/hr",
            f"Avg. Fulfillment Time = {sim_results['avg_fulfillment_time']:.2f} s",
            f"Avg. Wait Time = {sim_results['avg_wait']:.2f} s",
            f"Avg. Queue Length = {sim_results['avg_queue_delay']:.2f} orders",
            f"Total Cost = {sim_results['total_cost']:.1f}",
            f"Storage Policy: {STORAGE_POLICIES[current_policy]['name']}",
            f"Class Distribution: A={class_percentages['A']:.1f}%, B={class_percentages['B']:.1f}%, C={class_percentages['C']:.1f}%"
        ]
        
        if bool(int(slider_batch.val)):
            text_lines.append(f"Avg. Batch Size = {sim_results['avg_batch']:.2f}")
        
        ax_text.axis("off")
        ax_text.text(0.0, 1.0, "\n".join(text_lines), va="top", ha="left")
        fig.canvas.draw_idle()
    
    def set_system_type_ul(event):
        nonlocal storage_system_type
        storage_system_type = "unit_load"
        update()
    
    def set_system_type_ml(event):
        nonlocal storage_system_type
        storage_system_type = "miniload"
        update()
    
    def set_system_type_sb(event):
        nonlocal storage_system_type
        storage_system_type = "shuttle_based"
        update()
    
    def set_system_type_cb(event):
        nonlocal storage_system_type
        storage_system_type = "cube_based"
        update()
    
    btn_ul.on_clicked(set_system_type_ul)
    btn_ml.on_clicked(set_system_type_ml)
    btn_sb.on_clicked(set_system_type_sb)
    btn_cb.on_clicked(set_system_type_cb)
    
    slider_cranes.on_changed(update)
    slider_shuttles.on_changed(update)
    slider_robots.on_changed(update)
    slider_aisles.on_changed(update)
    slider_deep.on_changed(update)
    slider_command.on_changed(update)
    slider_peak.on_changed(update)
    slider_storage.on_changed(update)
    slider_batch.on_changed(update)
    slider_display.on_changed(update)
    policy_radio.on_clicked(update)
    
    update()
    plt.show()

# ==============================
# Analysis Functions for ABC Curve Evaluation
# ==============================
def analyze_abc_storage_impact():
    """
    Analyze the impact of different ABC storage policies on warehouse performance
    with enhanced visualizations for ABC curves, pick distribution, and pick density
    """
    policies = ["20_70", "20_90", "20_50", "20_20"]
    storage_types = ["unit_load", "miniload", "shuttle_based", "cube_based"]
    
    results = {}
    sku_distributions = {}
    
    for policy in policies:
        policy_results = {}
        
        # Update global product types for this policy
        global PRODUCT_TYPES
        PRODUCT_TYPES = generate_product_types(policy)
        
        for storage_type in storage_types:
            throughput, _, total_cost, _, avg_fulfillment, avg_wait, _, _, _, warehouse = run_simulation(
                storage_system_type=storage_type,
                num_cranes=10,
                num_shuttles=10,
                num_robots=10,
                num_aisles=10,
                cross_docking_rate=CROSS_DOCK_COST_RATE,
                peak_multiplier=1.0,
                deep_config="single",
                command_cycle="single",
                dynamic_storage=True,
                batch_mode=True,
                storage_policy=policy
            )
            
            policy_results[storage_type] = {
                "throughput": throughput,
                "total_cost": total_cost,
                "avg_fulfillment": avg_fulfillment,
                "avg_wait": avg_wait,
                "efficiency": throughput / (total_cost + 0.001)  # Avoid division by zero
            }
            
            # Store SKU distribution data for the first storage type
            if storage_type == storage_types[0]:
                sku_counts = warehouse.sku_counts
                sku_distributions[policy] = sku_counts
                
        results[policy] = policy_results
    
    # Create visualization - 2x2 grid of metrics comparison across storage types
    fig1, axes1 = plt.subplots(2, 2, figsize=(15, 10))
    axes1 = axes1.flatten()
    
    metrics = ["throughput", "avg_fulfillment", "avg_wait", "total_cost"]
    titles = ["Throughput (orders/hr)", "Average Fulfillment Time (s)", 
              "Average Wait Time (s)", "Total Cost"]
    
    for i, metric in enumerate(metrics):
        ax = axes1[i]
        
        # Data preparation
        x = np.arange(len(storage_types))
        width = 0.2
        
        # Plot bars for each policy
        for j, policy in enumerate(policies):
            policy_data = [results[policy][storage_type][metric] for storage_type in storage_types]
            ax.bar(x + j*width - 0.3, policy_data, width, label=STORAGE_POLICIES[policy]["name"])
        
        ax.set_title(titles[i])
        ax.set_xticks(x)
        ax.set_xticklabels(storage_types)
        ax.legend()
    
    plt.tight_layout()
    plt.savefig('abc_storage_metrics_comparison.png')
    
    # Create a new figure for ABC curve and pick distribution analysis
    fig2, axes2 = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Theoretical ABC curves
    ax_theory = axes2[0, 0]
    for policy in policies:
        # Theoretical curves
        x_values = np.linspace(0, 1, 100)
        if policy == "20_70":
            y_values = np.power(x_values, 0.2)  # 20% items account for 70% activity
        elif policy == "20_90":
            y_values = np.power(x_values, 0.1)  # 20% items account for 90% activity
        elif policy == "20_50":
            y_values = np.power(x_values, 0.4)  # 20% items account for 50% activity
        else:  # 20_20
            y_values = x_values  # Linear distribution
        
        ax_theory.plot(x_values, y_values, label=STORAGE_POLICIES[policy]["name"])
    
    ax_theory.set_title('Theoretical ABC Curves')
    ax_theory.set_xlabel('Cumulative % of Items')
    ax_theory.set_ylabel('Cumulative % of Activity')
    ax_theory.legend()
    ax_theory.grid(True, linestyle='--', alpha=0.7)
    
    # Plot 2: Actual Pick Distribution by Product Class
    ax_actual = axes2[0, 1]
    
    x = np.arange(len(policies))
    width = 0.25
    
    for policy in policies:
        sku_counts = sku_distributions[policy]
        total_picks = sum(sku_counts.values())
        
        class_totals = {'A': 0, 'B': 0, 'C': 0}
        for sku, count in sku_counts.items():
            product_class = PRODUCT_TYPES[sku]["class"]
            class_totals[product_class] += count
        
        class_percentages = [
            class_totals['A'] / total_picks * 100 if total_picks > 0 else 0,
            class_totals['B'] / total_picks * 100 if total_picks > 0 else 0,
            class_totals['C'] / total_picks * 100 if total_picks > 0 else 0
        ]
        
        # Find policy index
        policy_idx = policies.index(policy)
        
        # Plot actual class distribution as stacked bar
        bottom = 0
        for i, (cls, percentage) in enumerate(zip(['A', 'B', 'C'], class_percentages)):
            ax_actual.bar(policy_idx, percentage, width, bottom=bottom, 
                         label=f'Class {cls}' if policy_idx == 0 else "", 
                         color=['red', 'green', 'blue'][i], alpha=0.7)
            bottom += percentage
    
    ax_actual.set_title('Actual Pick Distribution by Product Class')
    ax_actual.set_ylabel('Percentage of Total Picks')
    ax_actual.set_xticks(x)
    ax_actual.set_xticklabels([STORAGE_POLICIES[p]["name"] for p in policies])
    ax_actual.legend()
    
    # Plot 3: Pick Density by Product Class across Aisles
    ax_density = axes2[1, 0]
    
    for policy_idx, policy in enumerate(policies):
        # Generate theoretical pick density based on policy
        num_aisles = 10
        aisle_density = []
        
        # A class (first 20% of aisles)
        a_aisles = max(1, int(0.2 * num_aisles))
        if policy == "20_70":
            a_density = 0.70 / a_aisles
        elif policy == "20_90":
            a_density = 0.90 / a_aisles
        elif policy == "20_50":
            a_density = 0.50 / a_aisles
        else:  # 20_20
            a_density = 0.20 / a_aisles
            
        for _ in range(a_aisles):
            aisle_density.append(a_density)
            
        # B class (next 30% of aisles)
        b_aisles = max(1, int(0.3 * num_aisles))
        if policy == "20_70":
            b_density = 0.20 / b_aisles
        elif policy == "20_90":
            b_density = 0.08 / b_aisles
        elif policy == "20_50":
            b_density = 0.30 / b_aisles
        else:  # 20_20
            b_density = 0.35 / b_aisles
            
        for _ in range(b_aisles):
            aisle_density.append(b_density)
            
        # C class (remaining aisles)
        c_aisles = num_aisles - a_aisles - b_aisles
        if policy == "20_70":
            c_density = 0.10 / c_aisles
        elif policy == "20_90":
            c_density = 0.02 / c_aisles
        elif policy == "20_50":
            c_density = 0.20 / c_aisles
        else:  # 20_20
            c_density = 0.45 / c_aisles
            
        for _ in range(c_aisles):
            aisle_density.append(c_density)
            
        # Pad if needed
        while len(aisle_density) < num_aisles:
            aisle_density.append(0)
            
        # Normalize to percentage
        aisle_density = [d * 100 for d in aisle_density]
            
        ax_density.plot(range(num_aisles), aisle_density, 
                       label=STORAGE_POLICIES[policy]["name"], 
                       marker='o', alpha=0.7)
    
    ax_density.set_title('Pick Density by Aisle (Class-based Storage)')
    ax_density.set_xlabel('Aisle Number (distance from I/O point)')
    ax_density.set_ylabel('Pick Density (%)')
    ax_density.legend()
    ax_density.grid(True, linestyle='--', alpha=0.5)
    
    # Plot 4: Actual SKU Activity Distribution
    ax_skus = axes2[1, 1]
    
    for policy_idx, policy in enumerate(policies):
        sku_data = sku_distributions[policy]
        # Sort SKUs by count
        sorted_skus = sorted(sku_data.items(), key=lambda x: x[1], reverse=True)
        skus = [item[0] for item in sorted_skus]
        counts = [item[1] for item in sorted_skus]
        
        # Convert to cumulative percentage
        total = sum(counts)
        cum_percent_skus = [i * 100 / len(skus) for i in range(1, len(skus) + 1)]
        cum_percent_counts = [sum(counts[:i+1]) * 100 / total for i in range(len(counts))]
        
        # Plot cumulative curve
        ax_skus.plot(cum_percent_skus, cum_percent_counts, 
                    label=STORAGE_POLICIES[policy]["name"], 
                    marker='x', alpha=0.7)
    
    # Add diagonal line (equal distribution)
    ax_skus.plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Equal Distribution')
    
    ax_skus.set_title('Actual SKU Activity Distribution')
    ax_skus.set_xlabel('Cumulative % of SKUs')
    ax_skus.set_ylabel('Cumulative % of Picks')
    ax_skus.legend()
    ax_skus.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig('abc_storage_distribution_analysis.png')
    plt.show()
    
    return results, sku_distributions

# ==============================
# Main Execution
# ==============================
if __name__ == "__main__":
    # Uncomment one of the following to run specific functions
    create_interactive_dashboard()
    analyze_abc_storage_impact()

## Fin.